# Confidence ablation

Tables and charts for one finished sweep of `scripts/run_ablation.py`.

```bash
python scripts/run_ablation.py --device 0            # run the sweep
python scripts/run_ablation.py --device 0 --resume   # continue an interrupted one
```

Current sweeps change the tracker high-score and track-birth thresholds together while preserving
low-score detector input. Dev sequences, counting rules and deterministic settings are shared.
The charts label the protocol recorded by each saved sweep, including historical detector-threshold sweeps. Latency is detector + tracker time per frame, without
image reading and report writing.

By default, read `outputs/ablation/confidence_dev` when its manifest exists. Otherwise, use the
only completed confidence sweep directly under `outputs/`. The selected folder and recorded
protocol are printed below. If there are several sweeps, choose one explicitly before running
the setup cell:

```python
import os
os.environ["ABLATION_DIR"] = "outputs/your_sweep_folder"  # absolute paths also work
```

An explicit path is never silently replaced. Every selected sweep still passes `validate_sweep`.
Selecting an archived sweep reads its saved results; it does not rerun the current pipeline.
The recorded NMS policy is also printed, because an older sweep may use another NMS.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Find the project from its own folders or the parent refactor workspace
def find_root(start):
    for directory in (start, *start.parents):
        for candidate in (directory, directory / "VehicleDetectionAndTrafficCounting"):
            if (candidate / "src/config.py").is_file() and (candidate / "pyproject.toml").is_file():
                return candidate
    raise FileNotFoundError(f"Repository root not found from {start}. Open the notebook inside refactor or the project.")

ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
from src.utils import file_info, read_json, write_json
from src.eval.validation import validate_sweep

def resolve_ablation_dir(root, override=None):
    """Use an explicit/default path, or the sole completed sweep in outputs/."""
    default = root / "outputs/ablation/confidence_dev"
    if override:
        selected = (root / Path(override).expanduser()).resolve()
        if not (selected / "manifest.json").is_file():
            raise FileNotFoundError(f"ABLATION_DIR has no manifest.json: {selected}")
        return selected
    if (default / "manifest.json").is_file():
        return default.resolve()  # validate_sweep will reject incomplete or changed runs

    candidates = []
    for manifest_path in sorted((root / "outputs").glob("*/manifest.json")):
        try:
            saved = read_json(manifest_path)
        except (OSError, ValueError):
            continue
        complete = saved.get("status") == "complete" if "status" in saved else saved.get("complete") is True
        plan = saved.get("plan", {})
        if complete and plan.get("jobs") and plan.get("confidences"):
            candidates.append(manifest_path.parent.resolve())
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        choices = "\n".join(f"  {p}" for p in candidates)
        raise ValueError("Several completed sweeps exist. Set os.environ['ABLATION_DIR'] "
                         f"to one of these folders and rerun this cell:\n{choices}")
    raise FileNotFoundError("No saved confidence sweep was found. Run "
                            "python scripts/run_ablation.py --device 0 from the repository root, "
                            "or set os.environ['ABLATION_DIR'] to an existing sweep.")


ABLATION_DIR = resolve_ablation_dir(ROOT, os.environ.get("ABLATION_DIR"))
print(f"Selected sweep: {ABLATION_DIR}")
OUTPUT_DIR = ABLATION_DIR / "analysis"
CLASSES = ["car", "bus", "truck", "all"]
plt.style.use("seaborn-v0_8-whitegrid")

## Check the sweep

The tables are built only from a complete sweep whose files still match the hashes in the manifest,
so a partial or edited experiment cannot pass as a full comparison.

In [ ]:
manifest = validate_sweep(ABLATION_DIR)
plan = manifest["plan"]
sequences, confidences, jobs = plan["sequences"], plan["confidences"], plan["jobs"]
print(f"{len(jobs)} runs, {len(sequences)} dev sequences, confidences {confidences}")
print(f"Environment: {manifest['tracking_environment']}")

## Tables

One row per confidence and configuration.
MAE: mean absolute count error per sequence, in vehicles.
WAPE: sum of absolute errors / sum of GT counts, in percent (n/a when the class has no GT vehicles).
SCE: predicted minus GT count, so negative means undercounting.
HOTA, MOTA and IDF1 are the dataset-level values TrackEval computes from all sequences together.

In [ ]:
LABELS = {"bytetrack": "ByteTrack", "botsort": "BoT-SORT"}
summary_rows, class_rows, sequence_rows = [], [], []
protocols, nms_policies = set(), set()

for confidence in confidences:
    reports = ABLATION_DIR / f"conf{confidence:g}/reports"
    counting = read_json(reports / "counting_eval_dev.json")
    tracking = read_json(reports / "tracking_eval_dev.json")

    for job in (j for j in jobs if j["confidence"] == confidence):
        config = job["config"]
        timing = read_json(ABLATION_DIR / f"conf{confidence:g}/tracks/dev/{config}/timing.json")
        approach = f"{Path(job['model']).stem.replace('yolov8', 'YOLOv8')} + {LABELS[job['tracker']]}"
        nms_policies.add({True: "class-agnostic", False: "class-aware"}.get(timing.get("agnostic_nms"), "not recorded"))
        settings = timing.get("tracker_settings", {})
        low_score_input = ("detector_conf" in timing and settings.get("track_high_thresh") == confidence
                           and settings.get("new_track_thresh") == confidence
                           and timing["detector_conf"] == min(settings.get("track_low_thresh", 0.1), confidence))
        protocols.add("tracker-threshold" if low_score_input else "detector-threshold")
        key = {"Approach": approach, "Config": config, "Confidence": confidence}

        for cls in CLASSES:
            r = counting[config][cls]       # pred, gt and sce hold one number per sequence, in plan order
            class_rows.append({**key, "Class": cls, "GT total": sum(r["gt"]), "Pred total": sum(r["pred"]),
                               "MAE": r["mae"], "WAPE (%)": None if r["wape"] is None else 100 * r["wape"],
                               "Mean SCE": r["mean_sce"]})
            sequence_rows += [{**key, "Class": cls, "Sequence": seq, "GT": gt, "Pred": pred, "SCE": pred - gt,
                               "Absolute error": abs(pred - gt)}
                              for seq, gt, pred in zip(sequences, r["gt"], r["pred"], strict=True)]

        # The last class row is "all": the summary adds the tracking and latency columns to it
        combined = tracking[config]["aggregate"]["combined"]
        summary_rows.append({**class_rows[-1], "HOTA (%)": combined["HOTA"], "MOTA (%)": combined["MOTA"],
                             "IDF1 (%)": combined["IDF1"], "IDSW": tracking[config]["aggregate"]["sum"]["IDSW"],
                             "Detector ms": timing["det_ms_per_frame"], "Tracker ms": timing["trk_ms_per_frame"],
                             "Latency ms": timing["total_ms_per_frame"], "FPS": timing["fps"]})

summary, per_class, per_sequence = map(pd.DataFrame, (summary_rows, class_rows, sequence_rows))
report_table = summary[["Confidence", "Approach", "MAE", "WAPE (%)", "IDF1 (%)", "Latency ms"]]
print(report_table.round(3).to_markdown(index=False))
if len(protocols) != 1:
    raise ValueError("The sweep mixes inference protocols")
CONFIDENCE_LABEL = ("Detector confidence (tracker thresholds fixed)" if protocols == {"detector-threshold"}
                    else "Tracker high/new threshold (low-score detector input)")
print("Recorded inference protocol:", next(iter(protocols)))

NMS_LABEL = ", ".join(sorted(nms_policies))
print("Recorded NMS:", NMS_LABEL)
from src.config import TRACK_AGNOSTIC_NMS
current_nms = "class-agnostic" if TRACK_AGNOSTIC_NMS else "class-aware"
if nms_policies != {current_nms}:
    print(f"Saved NMS differs from current project ({current_nms}). These are archived results; "
          "run a new sweep to evaluate the current pipeline.")


## Charts

Compare points on one curve to see the effect of confidence on one pipeline, and curves at the same
confidence to compare detectors and trackers. The zero line in the SCE chart separates overcounting
from undercounting. The heatmap shows whether one busy sequence explains a change.

In [ ]:
CHARTS_DIR, TABLES_DIR = OUTPUT_DIR / "charts", OUTPUT_DIR / "tables"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


def plot_curves(axis, column, title, zero=False):
    """One curve per detector + tracker pair, confidence on the x axis."""
    for approach, part in summary.groupby("Approach", sort=False):
        axis.plot(part["Confidence"], part[column], marker="o", label=approach)
    axis.set(title=title, xlabel=CONFIDENCE_LABEL, ylabel=column, xticks=confidences)
    if zero:
        axis.axhline(0, color="black", linewidth=1)
    axis.legend(fontsize=8)


def save_figure(figure, name):
    """Save as PNG and as vector PDF (for the report), then show it."""
    figure.tight_layout()
    for extension in ("png", "pdf"):
        figure.savefig(CHARTS_DIR / f"{name}_dev.{extension}", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(figure)


# Two panels per figure; the y axis starts at 0 because these values cannot be negative
PANELS = {"counting_sensitivity": [("Counting error", "WAPE (%)"), ("Error in vehicles", "MAE")],
          "tracking_sensitivity": [("Identity quality", "IDF1 (%)"), ("Identity switches", "IDSW")],
          "latency_sensitivity": [("Detector time", "Detector ms"), ("Tracker time", "Tracker ms")]}
for name, panels in PANELS.items():
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    for axis, (title, column) in zip(axes, panels, strict=True):
        plot_curves(axis, column, title)
        axis.set_ylim(bottom=0)
    save_figure(figure, name)

figure, axis = plt.subplots(figsize=(9, 5))
plot_curves(axis, "Mean SCE", "Direction of the counting error", zero=True)
save_figure(figure, "signed_counting_error")

# Signed error per sequence and condition, centred on 0 so over- and undercounting get opposite colours
errors = per_sequence[per_sequence["Class"] == "all"].assign(
    Condition=lambda d: d["Approach"] + " conf " + d["Confidence"].map("{:g}".format))
grid = errors.pivot(index="Condition", columns="Sequence", values="SCE")
grid = grid.reindex(index=errors["Condition"].drop_duplicates(), columns=sequences)
limit = max(1, float(np.abs(grid.to_numpy()).max()))
figure, axis = plt.subplots(figsize=(11, max(4, 0.4 * len(grid))))
sns.heatmap(grid, annot=True, fmt=".0f", cmap="RdBu_r", center=0, vmin=-limit, vmax=limit, ax=axis,
            cbar_kws={"label": "Predicted minus GT count"})
axis.set_title("Counting error per dev sequence")
save_figure(figure, "sequence_errors")

## Export

CSV keeps full precision. Markdown and LaTeX use three decimals and show undefined values as n/a.

Questions for the write-up:
- Does a lower confidence help every sequence or only some?
- Do better identity metrics also reduce the counting error?
- Does the detector or the tracker change the shape of the curve?
- Does a small total error hide a large error in one class?

In [ ]:
for name, table in (("ablation_summary", summary), ("ablation_report", report_table),
                    ("ablation_per_class", per_class), ("ablation_per_sequence", per_sequence)):
    table.to_csv(TABLES_DIR / f"{name}_dev.csv", index=False)

    # Text tables: floats with three decimals, integers unchanged
    text = table.astype(object)
    for column in table.select_dtypes("floating"):
        text[column] = table[column].map(lambda v: "n/a" if pd.isna(v) else f"{v:.3f}")
    (TABLES_DIR / f"{name}_dev.md").write_text(text.to_markdown(index=False, disable_numparse=True) + "\n",
                                               encoding="utf-8")
    (TABLES_DIR / f"{name}_dev.tex").write_text(text.to_latex(index=False, escape=True), encoding="utf-8")
print(f"Saved tables and charts to {OUTPUT_DIR}")

generated = sorted([*TABLES_DIR.glob("*"), *CHARTS_DIR.glob("*")])
write_json(OUTPUT_DIR / "analysis_manifest.json", {
    "experiment_manifest": file_info(ABLATION_DIR / "manifest.json"),
    "notebook": file_info(ROOT / "notebooks/ablation.ipynb"),
    "tracking_aggregation": "combined",
    "inference_protocol": sorted(protocols),
    "recorded_nms": sorted(nms_policies),
    "outputs": [file_info(p) for p in generated if p.is_file()],
})
